# NB0 — Multi-Region Data Merging and Cleaning
**Zombie Firms Replication** — *Geographical Analysis of the Italian Industrial North: Zombie Firms and Spillover Effects*

Advisor ref: Prof. Roberto Perotti (Bocconi, original thesis 2023–2024)

**Purpose:** Read all Orbis batch exports across all downloaded region subfolders, deduplicate firms within and across regions by BvD ID, merge into a single national panel, compute derived variables, and write clean Parquet files for NB1 (zombie classification).

This notebook replaces the Lombardia-only NB0. It is otherwise identical in parsing logic (same XML reader, variable map, derived variable formulas) so NB1–NB3 require no changes.

---
### Outputs
| File | Description |
|---|---|
| `zombie_panel_wide.parquet` | Wide format — one row per unique firm |
| `zombie_panel_long.parquet` | Long format — one row per firm-year (2008–2024) |
| `nb0_coverage_report.csv` | Missingness by variable and year |
| `nb0_merge_audit.csv` | Per-file row counts and region source |
| `nb0_cross_region_duplicates.csv` | BvD IDs appearing in more than one region folder (written only if duplicates detected) |
| `nb0_log.txt` | Full run log |

### Deduplication logic
- **Within region:** batch-boundary duplicates (same firm split across two Orbis export files) removed by BvD ID, keeping first occurrence.
- **Cross-region:** REGION_FOLDERS is ordered with named regions before `Others`. A firm appearing in both its home-region folder and `Others` retains the home-region row.

### Known structural issues handled
- **Loans & short-term debt**: years 2011–2014 appear as orphan columns at the end of the sheet. Merged by year label, not position.
- **2025 columns** (Tangible FA, Total assets): mostly empty, dropped by default.
- Duplicate `Company name` column (capitalisation difference): second copy dropped.
- `openpyxl` stylesheet bug (`CellStyle.__init__() got unexpected keyword argument 'applyNumFmt'`): all exports read via direct XML parsing, bypassing openpyxl entirely.

In [1]:
from pathlib import Path

# ── SETTINGS ──────────────────────────────────────────────────────────────────

# Root Data folder containing all region subfolders
EXPORT_DIR = Path("/Users/leoss/Desktop/Thesis Replication/Data")

# Output folder (created if it does not exist)
OUTPUT_DIR = Path("/Users/leoss/Desktop/Thesis Replication/output")

# Years to include in the long-format panel
PANEL_YEARS = list(range(2008, 2025))

# Drop the 2025 columns (Tangible FA 2025, Total assets 2025).
# These are mostly empty as of the April 2026 data update.
DROP_2025_COLS = True

# Consolidation code filter.
# "ALL"            → keep everything (recommended; filter in NB1)
# "UNCONSOLIDATED" → keep U1 and U2 only
# "CONSOLIDATED"   → keep C1 and C2 only
CONSOLIDATION_FILTER = "ALL"

# Region subfolders to scan, in order.
# Named regions come before 'Others' so that cross-region dedup retains
# the home-region row when a firm appears in both.
# Folders that do not exist are skipped silently.
REGION_FOLDERS = [
    "Lombardia", "Veneto", "Emilia-Romagna", "Piemonte",
    "Toscana", "FVG", "Liguria", "Lazio", "Marche", "Campania",
    "Abruzzo", "TAA", "Sicilia", "Puglia",
    "Sardegna", "Calabria", "Basilicata", "Molise", "Umbria", "Valle d'Aosta",
    "Others",
]

# ── END SETTINGS ──────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export dir  : {EXPORT_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Panel years : {PANEL_YEARS[0]}–{PANEL_YEARS[-1]}")
print(f"Consol mode : {CONSOLIDATION_FILTER}")

Export dir  : /Users/leoss/Desktop/Thesis Replication/Data
Output dir  : /Users/leoss/Desktop/Thesis Replication/output
Panel years : 2008–2024
Consol mode : ALL


In [2]:
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import pandas as pd

## XML-based Orbis reader
Bypasses the `openpyxl` stylesheet bug by parsing the xlsx ZIP directly. Orbis exports data on `sheet2`; if that is absent the reader falls back to `sheet1`.

In [3]:
NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'


def _col_letter_to_idx(ref: str) -> int:
    """Convert Excel cell reference like 'AB12' to 0-indexed column number."""
    letters = ''.join(c for c in ref if c.isalpha())
    n = 0
    for c in letters:
        n = n * 26 + (ord(c) - ord('A') + 1)
    return n - 1


def read_orbis_xlsx(path: Path) -> pd.DataFrame:
    """
    Read a single Orbis export .xlsx via direct XML parsing.
    Returns a DataFrame with original Orbis column names.
    Missing values ('n.a.' and empty strings) are replaced with NaN.
    Falls back to sheet1 if sheet2 is not present.
    """
    with zipfile.ZipFile(path) as z:
        with z.open('xl/sharedStrings.xml') as f:
            ss_root = ET.parse(f).getroot()
        strings = []
        for si in ss_root:
            t_els = si.findall(f'.//{NS}t')
            strings.append(''.join(t.text or '' for t in t_els))

        sheet_name = 'xl/worksheets/sheet2.xml'
        if sheet_name not in z.namelist():
            sheet_name = 'xl/worksheets/sheet1.xml'
        with z.open(sheet_name) as f:
            ws_root = ET.parse(f).getroot()

    # Header row
    headers = []
    for row in ws_root.iter(f'{NS}row'):
        if int(row.get('r', 0)) > 1:
            break
        for cell in row:
            t    = cell.get('t', '')
            v_el = cell.find(f'{NS}v')
            v    = v_el.text if v_el is not None else ''
            headers.append(strings[int(v)] if t == 's' and v else (v or ''))

    n_cols = len(headers)

    # Data rows
    records = []
    for row in ws_root.iter(f'{NS}row'):
        if int(row.get('r', 0)) == 1:
            continue
        row_vals = [''] * n_cols
        for cell in row:
            idx  = _col_letter_to_idx(cell.get('r', ''))
            t    = cell.get('t', '')
            v_el = cell.find(f'{NS}v')
            v    = v_el.text if v_el is not None else ''
            if t == 's' and v:
                v = strings[int(v)]
            if idx < n_cols:
                row_vals[idx] = v
        records.append(row_vals)

    df = pd.DataFrame(records, columns=headers)
    df.replace({'n.a.': np.nan, '': np.nan}, inplace=True)
    return df

## Column utilities

In [4]:
def deduplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only the first occurrence of each column name (case-insensitive)."""
    seen = {}
    keep = []
    for i, col in enumerate(df.columns):
        key = str(col).lower().strip()
        if key not in seen:
            seen[key] = i
            keep.append(i)
    return df.iloc[:, keep]


def normalise_column_name(col: str):
    """
    Parse 'Operating revenue (Turnover) th EUR 2022' → ('Operating revenue (Turnover)', 2022)
    Parse 'BvD ID number'                            → ('BvD ID number', None)
    """
    col = col.strip()
    m = re.search(r'\b(20\d{2})\s*$', col)
    if m:
        year = int(m.group(1))
        var  = re.sub(r'\s+th EUR\s*$', '', col[:m.start()]).strip()
        return var, year
    return col, None


def build_variable_map(df: pd.DataFrame):
    """
    Returns:
        time_vars  : {variable_name: {year: column_index}}
        static_cols: [(column_index, column_name)]
    """
    time_vars   = defaultdict(dict)
    static_cols = []
    for i, col in enumerate(df.columns):
        if not col or pd.isna(col):
            continue
        var, year = normalise_column_name(col)
        if year is not None:
            time_vars[var][year] = i
        else:
            static_cols.append((i, col))
    return dict(time_vars), static_cols

## Reshape: wide → long

In [5]:
def wide_to_long(df_wide: pd.DataFrame,
                 time_vars: dict,
                 static_cols: list,
                 panel_years: list) -> pd.DataFrame:
    """
    Convert wide Orbis DataFrame to long panel (one row per firm-year).
    Years absent from the variable map for a given variable are filled with NaN.
    """
    static_idx   = [i for i, _ in static_cols]
    static_names = [n for _, n in static_cols]
    df_static = df_wide.iloc[:, static_idx].copy()
    df_static.columns = static_names
    df_static['_row'] = range(len(df_static))

    frames = []
    for year in panel_years:
        yd = df_static.copy()
        yd['year'] = year
        for var, year_map in time_vars.items():
            yd[var] = df_wide.iloc[:, year_map[year]].values if year in year_map else np.nan
        frames.append(yd)

    df_long = pd.concat(frames, ignore_index=True)
    df_long.drop(columns=['_row'], inplace=True)
    return df_long

## Rename map, numeric parsing, and derived variables

In [6]:
def parse_numeric(s: pd.Series) -> pd.Series:
    """Convert Orbis financial string columns (comma thousands separators) to float."""
    return pd.to_numeric(s.astype(str).str.replace(',', '', regex=False), errors='coerce')


RENAME_MAP = {
    'Operating profit (loss) [EBIT]'              : 'ebit',
    'Financial expenses'                           : 'financial_expenses',
    'P/L for period [=Net income]'                 : 'net_income',
    'Total assets'                                 : 'total_assets',
    'Shareholders funds'                           : 'shareholders_funds',
    'Tangible fixed assets'                        : 'tangible_fa',
    'Operating revenue (Turnover)'                 : 'turnover',
    'Added value'                                  : 'added_value',
    'Costs of employees'                           : 'costs_employees',
    'Material costs'                               : 'material_costs',
    'EBITDA'                                       : 'ebitda',
    'Depreciation & Amortization'                  : 'dep_amort',
    'Current liabilities'                          : 'current_liabilities',
    'Non-current liabilities'                      : 'noncurrent_liabilities',
    'Loans & short-term debt'                      : 'loans_std',
    'Number of employees'                          : 'employees',
    'GUO - Name'                                   : 'guo_name',
    'GUO - BvD ID number'                          : 'guo_bvd_id',
    'GUO - Country ISO code'                       : 'guo_country',
    'SH - BvD Independence Indicator'              : 'independence_indicator',
    'Company name Latin alphabet'                  : 'company_name',
    'Address line 1 Latin Alphabet'                : 'address',
    'Postcode Latin Alphabet'                      : 'postcode',
    'City Latin Alphabet'                          : 'city',
    'Country'                                      : 'country',
    'Region in country'                            : 'region',
    'NUTS3'                                        : 'nuts3',
    'BvD ID number'                                : 'bvd_id',
    'Standardized legal form'                      : 'legal_form_std',
    'National legal form'                          : 'legal_form_nat',
    'Status'                                       : 'status',
    'Status date'                                  : 'status_date',
    'Date of incorporation'                        : 'date_incorporation',
    'Consolidation code'                           : 'consolidation_code',
    'Listing status'                               : 'listing_status',
    'NACE Rev. 2, primary code(s)'                 : 'nace_primary',
    'NACE Rev. 2, primary code(s) - description'   : 'nace_primary_desc',
    'NACE Rev. 2, secondary code(s)'               : 'nace_secondary',
    'NACE Rev. 2, secondary code(s) - description' : 'nace_secondary_desc',
    'BvD sectors'                                  : 'bvd_sectors',
}

NUMERIC_COLS = [
    'ebit', 'financial_expenses', 'net_income', 'total_assets',
    'shareholders_funds', 'tangible_fa', 'turnover', 'added_value',
    'costs_employees', 'material_costs', 'ebitda', 'dep_amort',
    'current_liabilities', 'noncurrent_liabilities', 'loans_std', 'employees',
]


def rename_and_parse(df: pd.DataFrame) -> pd.DataFrame:
    rename_actual = {k: v for k, v in RENAME_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_actual)
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = parse_numeric(df[col])
    return df


def compute_derived_variables(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derived variables:
      total_liabilities  = total_assets - shareholders_funds   (proxy; direct var has 3-4% coverage)
      icr                = ebit / financial_expenses            (NaN when financial_expenses <= 0)
      roa                = net_income / total_assets            (NaN when total_assets = 0)
      investment_rate    = delta(tangible_fa) / lagged total_assets
      neg_equity         = 1 if shareholders_funds < 0
      nace_2digit        = first 2 digits of nace_primary
    """
    if 'total_assets' in df.columns and 'shareholders_funds' in df.columns:
        df['total_liabilities'] = df['total_assets'] - df['shareholders_funds']

    if 'ebit' in df.columns and 'financial_expenses' in df.columns:
        fe = df['financial_expenses'].copy()
        fe[fe <= 0] = np.nan
        df['icr'] = df['ebit'] / fe

    if 'net_income' in df.columns and 'total_assets' in df.columns:
        ta = df['total_assets'].copy()
        ta[ta == 0] = np.nan
        df['roa'] = df['net_income'] / ta

    if 'tangible_fa' in df.columns and 'total_assets' in df.columns and 'bvd_id' in df.columns:
        df = df.sort_values(['bvd_id', 'year'])
        df['_tfa_lag'] = df.groupby('bvd_id')['tangible_fa'].shift(1)
        df['_ta_lag']  = df.groupby('bvd_id')['total_assets'].shift(1)
        valid_lag = df['_ta_lag'] > 0
        df['investment_rate'] = np.where(
            valid_lag,
            (df['tangible_fa'] - df['_tfa_lag']) / df['_ta_lag'],
            np.nan
        )
        df.drop(columns=['_tfa_lag', '_ta_lag'], inplace=True)

    if 'shareholders_funds' in df.columns:
        df['neg_equity'] = (df['shareholders_funds'] < 0).astype(float)
        df.loc[df['shareholders_funds'].isna(), 'neg_equity'] = np.nan

    if 'nace_primary' in df.columns:
        df['nace_2digit'] = pd.to_numeric(
            df['nace_primary'].astype(str).str[:2], errors='coerce'
        ).astype('Int64')

    return df

## Consolidation filter and coverage report

In [7]:
def apply_consolidation_filter(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    if mode == "ALL":
        return df
    if 'consolidation_code' not in df.columns:
        print("WARNING: consolidation_code not found, skipping filter.")
        return df
    keep_codes = {'UNCONSOLIDATED': ['U1', 'U2'], 'CONSOLIDATED': ['C1', 'C2']}.get(mode)
    if keep_codes is None:
        raise ValueError(f"Unknown consolidation mode: {mode}")
    before = df['bvd_id'].nunique() if 'bvd_id' in df.columns else len(df)
    df = df[df['consolidation_code'].isin(keep_codes)].copy()
    after  = df['bvd_id'].nunique() if 'bvd_id' in df.columns else len(df)
    print(f"Consolidation filter ({mode}): {before:,} → {after:,} unique firms")
    return df


KEY_VARS_COVERAGE = [
    'total_assets', 'ebit', 'financial_expenses', 'net_income',
    'shareholders_funds', 'tangible_fa', 'employees', 'loans_std',
    'icr', 'roa', 'investment_rate'
]


def coverage_report(df_long: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    """Coverage (share non-missing) by variable and year."""
    existing = [v for v in KEY_VARS_COVERAGE if v in df_long.columns]
    records  = []
    for year in sorted(df_long['year'].unique()):
        sub = df_long[df_long['year'] == year]
        n   = len(sub)
        row = {'year': year, 'n_firms': n}
        for var in existing:
            row[f'{var}_cov'] = sub[var].notna().sum() / n if n > 0 else np.nan
        records.append(row)
    report = pd.DataFrame(records)
    report.to_csv(out_path, index=False)
    return report

---
## Step 1 — Discover export files across region folders

In [8]:
t0 = time.time()
log = []

def L(msg):
    print(msg)
    log.append(str(msg))

L("=" * 65)
L("NB0 — Multi-Region Data Merging and Cleaning")
L(f"Export dir  : {EXPORT_DIR}")
L(f"Output dir  : {OUTPUT_DIR}")
L(f"Panel years : {PANEL_YEARS[0]}–{PANEL_YEARS[-1]}")
L(f"Consol mode : {CONSOLIDATION_FILTER}")
L("=" * 65)

L("\n--- Scanning region folders ---")
audit_rows = []
all_files  = []   # list of (region_label, Path)

for region in REGION_FOLDERS:
    folder = EXPORT_DIR / region
    if not folder.exists():
        L(f"  {region:<25s} → folder not found, skipping")
        continue
    files = sorted(folder.glob("Export_*.xlsx"))
    if not files:
        files = sorted(folder.glob("*.xlsx"))   # fallback for non-standard names
    if not files:
        L(f"  {region:<25s} → no .xlsx files found")
        continue
    L(f"  {region:<25s} → {len(files)} file(s)")
    for f in files:
        L(f"      {f.name}")
        all_files.append((region, f))

assert all_files, (
    f"No Export_*.xlsx files found under {EXPORT_DIR}.\n"
    "Check REGION_FOLDERS and that files are named Export_*.xlsx."
)
L(f"\nTotal files to read: {len(all_files)}")

NB0 — Multi-Region Data Merging and Cleaning
Export dir  : /Users/leoss/Desktop/Thesis Replication/Data
Output dir  : /Users/leoss/Desktop/Thesis Replication/output
Panel years : 2008–2024
Consol mode : ALL

--- Scanning region folders ---
  Lombardia                 → 8 file(s)
      Lombardia_1.xlsx
      Lombardia_2.xlsx
      Lombardia_3.xlsx
      Lombardia_4.xlsx
      Lombardia_5.xlsx
      Lombardia_6.xlsx
      Lombardia_7.xlsx
      Lombardia_8.xlsx
  Veneto                    → 5 file(s)
      Veneto_1.xlsx
      Veneto_2.xlsx
      Veneto_3.xlsx
      Veneto_4.xlsx
      Veneto_5.xlsx
  Emilia-Romagna            → 4 file(s)
      EmiliaRomagna_1.xlsx
      EmiliaRomagna_2.xlsx
      EmiliaRomagna_3.xlsx
      EmiliaRomagna_4.xlsx
  Piemonte                  → 3 file(s)
      Piemonte_1.xlsx
      Piemonte_2.xlsx
      Piemonte_4.xlsx
  Toscana                   → 3 file(s)
      Toscana_1.xlsx
      Toscana_2.xlsx
      Toscana_3.xlsx
  FVG                       → 1 file(s)

## Step 2 — Read and concatenate all exports

In [9]:
L("\n--- Reading exports ---")
dfs_wide = []

for region, f in all_files:
    try:
        df = read_orbis_xlsx(f)
        df = deduplicate_columns(df)
        df['_source_file']   = f.name
        df['_source_region'] = region
        n  = len(df)
        L(f"  [{region}] {f.name:<55s} {n:>5,} rows")
        audit_rows.append({'region': region, 'file': f.name, 'rows_raw': n})
        dfs_wide.append(df)
    except Exception as e:
        L(f"  ERROR [{region}] {f.name}: {e}")
        audit_rows.append({'region': region, 'file': f.name, 'rows_raw': 0, 'error': str(e)})

assert dfs_wide, "No files read successfully."

df_all = pd.concat(dfs_wide, ignore_index=True)
L(f"\nCombined (pre-dedup): {len(df_all):,} rows, {len(df_all.columns)} columns")


--- Reading exports ---
  [Lombardia] Lombardia_1.xlsx                                        3,058 rows
  [Lombardia] Lombardia_2.xlsx                                        3,058 rows
  [Lombardia] Lombardia_3.xlsx                                        3,058 rows
  [Lombardia] Lombardia_4.xlsx                                        3,058 rows
  [Lombardia] Lombardia_5.xlsx                                        3,058 rows
  [Lombardia] Lombardia_6.xlsx                                        3,058 rows
  [Lombardia] Lombardia_7.xlsx                                        3,058 rows
  [Lombardia] Lombardia_8.xlsx                                          489 rows
  [Veneto] Veneto_1.xlsx                                           3,058 rows
  [Veneto] Veneto_2.xlsx                                           3,058 rows
  [Veneto] Veneto_3.xlsx                                           3,058 rows
  [Veneto] Veneto_4.xlsx                                           3,058 rows
  [Veneto] Vene

/var/folders/lk/thldsylx4nx779cggs7gnk900000gn/T/ipykernel_29040/2017481076.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'n.a.': np.nan, '': np.nan}, inplace=True)


  [Emilia-Romagna] EmiliaRomagna_1.xlsx                                    3,058 rows
  [Emilia-Romagna] EmiliaRomagna_2.xlsx                                    3,058 rows
  [Emilia-Romagna] EmiliaRomagna_3.xlsx                                    3,058 rows
  [Emilia-Romagna] EmiliaRomagna_4.xlsx                                      625 rows
  [Piemonte] Piemonte_1.xlsx                                         3,058 rows
  [Piemonte] Piemonte_2.xlsx                                         3,058 rows
  [Piemonte] Piemonte_4.xlsx                                           613 rows
  [Toscana] Toscana_1.xlsx                                          3,058 rows
  [Toscana] Toscana_2.xlsx                                          3,058 rows
  [Toscana] Toscana_3.xlsx                                          2,289 rows
  [FVG] FVG.xlsx                                                2,072 rows
  [Liguria] Liguria_1.xlsx                                          3,058 rows
  [Liguria] Liguria_2.xls

## Step 3 — Drop 2025 columns

In [10]:
if DROP_2025_COLS:
    cols_2025 = [c for c in df_all.columns if str(c).strip().endswith('2025')]
    if cols_2025:
        df_all.drop(columns=cols_2025, inplace=True)
        L(f"Dropped 2025 columns: {cols_2025}")
    else:
        L("No 2025 columns found.")

Dropped 2025 columns: ['Tangible fixed assets th EUR 2025', 'Total assets th EUR 2025']


## Step 4 — Cross-region deduplication by BvD ID
Named regions come before `Others` in `REGION_FOLDERS`, so `keep='first'` retains the home-region row when a firm appears in multiple folders. Duplicates are written to `nb0_cross_region_duplicates.csv` for inspection.

In [11]:
bvd_col = next((c for c in df_all.columns if 'bvd id number' in c.lower()), None)

if bvd_col:
    before   = len(df_all)
    dup_mask = df_all.duplicated(subset=[bvd_col], keep='first')
    n_cross_dup = dup_mask.sum()

    if n_cross_dup > 0:
        dup_bvd = df_all.loc[dup_mask, [bvd_col, '_source_region']].copy()
        dup_summary = (
            dup_bvd.groupby(bvd_col)['_source_region']
            .apply(lambda x: ', '.join(sorted(set(x))))
            .reset_index()
        )
        dup_summary.columns = ['bvd_id', 'duplicate_regions']
        dup_path = OUTPUT_DIR / "nb0_cross_region_duplicates.csv"
        dup_summary.to_csv(dup_path, index=False)
        L(f"Cross-region duplicates: {n_cross_dup:,} rows removed")
        L(f"  → detail saved to {dup_path}")
    else:
        L("No cross-region duplicates detected.")

    df_all.drop_duplicates(subset=[bvd_col], keep='first', inplace=True)
    after = len(df_all)
    L(f"Deduplication by BvD ID: {before:,} → {after:,} firms "
      f"({before - after:,} total duplicates removed)")
else:
    L("WARNING: BvD ID column not found — skipping cross-region deduplication.")

L(f"Unique firms after deduplication: {len(df_all):,}")

Cross-region duplicates: 3,970 rows removed
  → detail saved to /Users/leoss/Desktop/Thesis Replication/output/nb0_cross_region_duplicates.csv
Deduplication by BvD ID: 90,599 → 86,629 firms (3,970 total duplicates removed)
Unique firms after deduplication: 86,629


## Step 5 — Build variable map

In [12]:
time_vars, static_cols = build_variable_map(df_all)
L(f"\nTime-varying variables : {len(time_vars)}")
L(f"Static / identifier cols: {len(static_cols)}")

# Loans gap check
if 'Loans & short-term debt' in time_vars:
    yrs_found   = sorted(time_vars['Loans & short-term debt'].keys())
    missing_yrs = [y for y in range(2008, 2025) if y not in yrs_found]
    if missing_yrs:
        L(f"NOTE — Loans: missing years {missing_yrs} (will be NaN in panel)")
    else:
        L("Loans: all years 2008–2024 present (orphan cols merged correctly)")


Time-varying variables : 16
Static / identifier cols: 26
Loans: all years 2008–2024 present (orphan cols merged correctly)


## Step 6 — Reshape to long format

In [13]:
L("\n--- Reshaping to long format ---")
df_long = wide_to_long(df_all, time_vars, static_cols, PANEL_YEARS)
L(f"Long panel: {len(df_long):,} firm-year observations")


--- Reshaping to long format ---
Long panel: 1,472,693 firm-year observations


## Step 7 — Rename columns and parse numerics

In [14]:
L("\n--- Renaming columns and parsing numerics ---")
df_long = rename_and_parse(df_long)
df_all  = rename_and_parse(df_all)


--- Renaming columns and parsing numerics ---


## Step 8 — Derived variables

In [15]:
L("\n--- Computing derived variables ---")
df_long = compute_derived_variables(df_long)

derived = ['total_liabilities', 'icr', 'roa', 'investment_rate', 'neg_equity', 'nace_2digit']
for v in derived:
    if v in df_long.columns:
        cov = df_long[v].notna().mean()
        L(f"  {v:<22s}: {cov:.1%} non-missing (all years pooled)")


--- Computing derived variables ---
  total_liabilities     : 47.8% non-missing (all years pooled)
  icr                   : 44.4% non-missing (all years pooled)
  roa                   : 47.8% non-missing (all years pooled)
  investment_rate       : 42.2% non-missing (all years pooled)
  neg_equity            : 47.8% non-missing (all years pooled)
  nace_2digit           : 100.0% non-missing (all years pooled)


## Step 9 — Consolidation filter

In [16]:
L("\n--- Consolidation filter ---")
df_long = apply_consolidation_filter(df_long, CONSOLIDATION_FILTER)


--- Consolidation filter ---


## Step 10 — Sanity checks

In [17]:
sub22 = df_long[df_long['year'] == 2022].copy()
L(f"\n2022 cross-section: {len(sub22):,} firms")

# ICR
if 'icr' in df_long.columns:
    icr = sub22['icr'].dropna()
    L(f"\nICR 2022 (n={len(icr):,}):")
    L(f"  Median          : {icr.median():.2f}")
    L(f"  Share ICR < 1   : {(icr < 1).mean():.1%}  (potential zombies under McGowan)")
    L(f"  Share ICR < 0   : {(icr < 0).mean():.1%}  (EBIT negative)")

# Negative equity
if 'neg_equity' in df_long.columns:
    ne = sub22['neg_equity'].dropna()
    L(f"\nNegative equity 2022: {ne.mean():.1%} (n={len(ne):,})")

# Region distribution
if 'region' in df_long.columns:
    L("\nRegion distribution (2022):")
    for reg, cnt in sub22['region'].value_counts().items():
        L(f"  {str(reg):<35s}: {cnt:>6,}")

# Consolidation codes
if 'consolidation_code' in df_long.columns:
    L("\nConsolidation codes (2022):")
    for cc, cnt in sub22['consolidation_code'].value_counts().items():
        L(f"  {cc}: {cnt:,}")

# Employee thresholds
if 'employees' in df_long.columns:
    emp = sub22['employees'].dropna()
    L(f"\nEmployee threshold check 2022 (n={len(emp):,}):")
    for thr in [10, 20, 50]:
        L(f"  >= {thr:2d} employees: {(emp >= thr).mean():.1%}")


2022 cross-section: 86,629 firms

ICR 2022 (n=64,930):
  Median          : 7.25
  Share ICR < 1   : 15.2%  (potential zombies under McGowan)
  Share ICR < 0   : 12.9%  (EBIT negative)

Negative equity 2022: 5.8% (n=69,297)

Region distribution (2022):
  Lombardia                          : 21,247
  Veneto                             : 12,116
  Emilia-Romagna                     :  9,492
  Toscana                            :  8,308
  Piemonte                           :  6,560
  Campania                           :  5,765
  Marche                             :  3,845
  Lazio                              :  3,785
  Puglia                             :  3,556
  Sicilia                            :  2,383
  Friuli-Venezia Giulia              :  2,016
  Abruzzo                            :  1,749
  Umbria                             :  1,293
  Liguria                            :  1,205
  Trentino-Alto Adige                :  1,160
  Calabria                           :    739
  Sardegna 

## Step 11 — Coverage report

In [18]:
L("\n--- Coverage report ---")
cov_df = coverage_report(df_long, OUTPUT_DIR / "nb0_coverage_report.csv")

core_years = list(range(2016, 2025))
disp_cols  = ['year', 'n_firms'] + [
    c for c in cov_df.columns
    if c.endswith('_cov') and any(k in c for k in ['total_assets', 'ebit', 'employees'])
]
print("\nCoverage (core panel 2016–2024):")
print(cov_df[cov_df['year'].isin(core_years)][disp_cols].to_string(index=False))


--- Coverage report ---

Coverage (core panel 2016–2024):
 year  n_firms  total_assets_cov  ebit_cov  employees_cov
 2016    86629          0.746574  0.746574       0.732087
 2017    86629          0.766279  0.766279       0.749899
 2018    86629          0.781043  0.781043       0.760565
 2019    86629          0.791825  0.791813       0.770054
 2020    86629          0.794307  0.794307       0.771462
 2021    86629          0.798774  0.798774       0.825971
 2022    86629          0.799905  0.799905       0.831177
 2023    86629          0.791109  0.791109       0.833335
 2024    86629          0.763209  0.763209       0.808182


## Step 12 — Save audit table

In [19]:
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUTPUT_DIR / "nb0_merge_audit.csv", index=False)
L(f"Saved audit table → {OUTPUT_DIR / 'nb0_merge_audit.csv'}")
print(audit_df.to_string(index=False))

Saved audit table → /Users/leoss/Desktop/Thesis Replication/output/nb0_merge_audit.csv
        region                 file  rows_raw
     Lombardia     Lombardia_1.xlsx      3058
     Lombardia     Lombardia_2.xlsx      3058
     Lombardia     Lombardia_3.xlsx      3058
     Lombardia     Lombardia_4.xlsx      3058
     Lombardia     Lombardia_5.xlsx      3058
     Lombardia     Lombardia_6.xlsx      3058
     Lombardia     Lombardia_7.xlsx      3058
     Lombardia     Lombardia_8.xlsx       489
        Veneto        Veneto_1.xlsx      3058
        Veneto        Veneto_2.xlsx      3058
        Veneto        Veneto_3.xlsx      3058
        Veneto        Veneto_4.xlsx      3058
        Veneto        Veneto_5.xlsx       202
Emilia-Romagna EmiliaRomagna_1.xlsx      3058
Emilia-Romagna EmiliaRomagna_2.xlsx      3058
Emilia-Romagna EmiliaRomagna_3.xlsx      3058
Emilia-Romagna EmiliaRomagna_4.xlsx       625
      Piemonte      Piemonte_1.xlsx      3058
      Piemonte      Piemonte_2.xlsx    

## Step 13 — Save outputs

In [20]:
# Wide (one row per firm)
wide_out = OUTPUT_DIR / "zombie_panel_wide.parquet"
df_all.to_parquet(wide_out, index=False)
L(f"\nSaved: {wide_out}  ({len(df_all):,} firms)")

# Long (one row per firm-year)
long_out = OUTPUT_DIR / "zombie_panel_long.parquet"
df_long.to_parquet(long_out, index=False)
L(f"Saved: {long_out}  ({len(df_long):,} obs)")

# Log
with open(OUTPUT_DIR / "nb0_log.txt", 'w') as f:
    f.write('\n'.join(log))

elapsed = time.time() - t0
L(f"\nDone in {elapsed:.1f}s")
L("=" * 65)

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"Regions found         : {audit_df['region'].nunique()}")
print(f"Files read            : {len(dfs_wide)}")
print(f"Unique firms (wide)   : {len(df_all):,}")
print(f"Firm-year obs (long)  : {len(df_long):,}")
print(f"Panel span            : {PANEL_YEARS[0]}–{PANEL_YEARS[-1]}")
print(f"Outputs saved to      : {OUTPUT_DIR}")
print("=" * 65)


Saved: /Users/leoss/Desktop/Thesis Replication/output/zombie_panel_wide.parquet  (86,629 firms)
Saved: /Users/leoss/Desktop/Thesis Replication/output/zombie_panel_long.parquet  (1,472,693 obs)

Done in 160.4s

SUMMARY
Regions found         : 15
Files read            : 39
Unique firms (wide)   : 86,629
Firm-year obs (long)  : 1,472,693
Panel span            : 2008–2024
Outputs saved to      : /Users/leoss/Desktop/Thesis Replication/output
